# Project: Grocery Shopping Patterns: A Transaction Analysis

**Objective:** Analyze grocery transaction data from two 24-hour stores to identify consumer spending patterns, shopping preferences, and purchasing behavior during the given time period.

**Business Questions:**

* Which week of the year had total sales closest to the mean weekly sales between June 1 and August 31?
* Which hour of the day generated the highest total sales?
* How many days passed between the three Cornflakes purchases made by `CustomerID` 107?

**Data Used:**

* `grocery_data1.csv`
* `grocery_data2.csv`

Both datasets contain:

* `CustomerID` : Unique customer identifier
* `DateRaw` : Raw transaction date
* `Time` : Time of the transaction
* `TransactionID` : Unique transaction identifier
* `ProductName` : Name of the product purchased
* `PriceUSD` : Price of the product in US dollars
* `Quantity` : Number of units purchased
* `PaymentMethod` : Payment method used
* `Category` : Product category

**Methodology:**

1. Import the required R packages and load both grocery transaction datasets.
2. Inspect the datasets and their column types to understand their structure.
3. Convert the raw transaction dates into the appropriate date format for each dataset, then create `Week` and `Year` variables.
4. Combine the two store datasets into a single transaction dataset and calculate total sales for each transaction using price and quantity.
5. Group transactions by `Week` and `Year` to calculate total weekly sales, then identify the week with the smallest absolute deviation from the mean weekly sales.
6. Convert transaction times into `hms` format and extract the hour, minute, and second components.
7. Group transactions by hour and calculate total sales to identify the hour with the highest sales.
8. Filter the data to `CustomerID` 107 and `ProductName` "Cornflakes", order the purchases chronologically, and calculate the number of days between consecutive purchases.
9. Extract the two relevant purchase intervals and store them as the `cornflakes_days` integer vector.

**Output Objects:**

* `smallest_sales_deviation` : Week number with the smallest absolute deviation from mean weekly sales
* `most_hourly_sales` : Hour of the day with the highest total sales
* `cornflakes_days` : Integer vector containing the number of days between the first and second, and second and third Cornflakes purchases by `CustomerID` 107


In [15]:
# Import packages
suppressPackageStartupMessages({
library(dplyr)
library(lubridate)
library(readr)
library(hms)})

# Load both files into grocery_1 and grocery_2
grocery_1 <- read.csv('grocery_data1.csv')
grocery_2 <- read.csv('grocery_data2.csv')

# View their output
head(grocery_1, n = 5)
cat('\n')
head(grocery_2, n = 5 )

,CustomerID,DateRaw,Time,TransactionID,ProductName,PriceUSD,Quantity,PaymentMethod,Category
,<int>,<chr>,<chr>,<int>,<chr>,<dbl>,<int>,<chr>,<chr>
1,41,"June 28, 2023",20:00:00,2,Apples,5.64,5,Cash,Produce
2,170,"August 18, 2023",06:00:00,3,Apples,17.92,1,Mobile Payment,Produce
3,86,"August 18, 2023",09:00:00,4,Pasta,19.14,2,Mobile Payment,Grains
4,178,"August 06, 2023",02:00:00,5,Rice,0.76,4,Debit Card,Grains
5,87,"July 30, 2023",09:00:00,6,Chickpeas,11.30,1,Debit Card,Vegetarian


,CustomerID,DateRaw,Time,TransactionID,ProductName,PriceUSD,Quantity,PaymentMethod,Category
,<int>,<chr>,<chr>,<int>,<chr>,<dbl>,<int>,<chr>,<chr>
1,164,22 July 2023,20:00:00,1,Beef,10.26,2,Cash,Meat
2,15,20 August 2023,08:00:00,7,Carrots,5.60,5,Debit Card,Produce
3,121,22 June 2023,00:00:00,10,Cornflakes,7.47,1,Debit Card,Cereal
4,91,30 June 2023,17:00:00,14,Milk,13.66,1,Cash,Dairy
5,148,28 August 2023,04:00:00,18,Chickpeas,10.43,1,Debit Card,Vegetarian


In [16]:
# Check column types
str(grocery_1)
cat("\n")
str(grocery_2)

'data.frame':	2619 obs. of  9 variables:
 $ CustomerID   : int  41 170 86 178 87 12 104 140 104 78 ...
 $ DateRaw      : chr  "June 28, 2023" "August 18, 2023" "August 18, 2023" "August 06, 2023" ...
 $ Time         : chr  "20:00:00" "06:00:00" "09:00:00" "02:00:00" ...
 $ TransactionID: int  2 3 4 5 6 8 9 11 12 13 ...
 $ ProductName  : chr  "Apples" "Apples" "Pasta" "Rice" ...
 $ PriceUSD     : num  5.64 17.92 19.14 0.76 11.3 ...
 $ Quantity     : int  5 1 2 4 1 2 2 2 1 1 ...
 $ PaymentMethod: chr  "Cash" "Mobile Payment" "Mobile Payment" "Debit Card" ...
 $ Category     : chr  "Produce" "Produce" "Grains" "Grains" ...

'data.frame':	2581 obs. of  9 variables:
 $ CustomerID   : int  164 15 121 91 148 93 121 50 142 128 ...
 $ DateRaw      : chr  "22 July 2023" "20 August 2023" "22 June 2023" "30 June 2023" ...
 $ Time         : chr  "20:00:00" "08:00:00" "00:00:00" "17:00:00" ...
 $ TransactionID: int  1 7 10 14 18 20 22 23 24 29 ...
 $ ProductName  : chr  "Beef" "Carrots" "Cornflakes"

In [17]:
# Step 1: Which week between June 1 and August 31 had the smallest absolute deviation in sales compared to the mean weekly sales for that same time period?

# Convert DateRaw from a character type to date. Drop DateRaw and use the new Date column to create Week and Year columns.
grocery_1 <- 
grocery_1 %>%
mutate(Date = mdy(DateRaw)) %>%
mutate(Week = week(Date)) %>%
mutate(Year = year(Date)) %>%
select(-DateRaw) %>%
relocate(Date, .after = CustomerID) %>%
relocate(Week, Year, .after = Time)

# Repeat the same for grocery_2 dataframe
grocery_2 <- 
grocery_2 %>%
mutate(Date = dmy(DateRaw)) %>%
mutate(Week = week(Date)) %>%
mutate(Year = year(Date)) %>%
select(-DateRaw) %>%
relocate(Date, .after = CustomerID) %>%
relocate(Week, Year, .after = Time)

# Combine both dataframes into a combined dataframe.
combined <- 
bind_rows(grocery_1, grocery_2)

# View output of combined
combined %>%
head(n=5)

,CustomerID,Date,Time,Week,Year,TransactionID,ProductName,PriceUSD,Quantity,PaymentMethod,Category
,<int>,<date>,<chr>,<dbl>,<dbl>,<int>,<chr>,<dbl>,<int>,<chr>,<chr>
1,41,2023-06-28,20:00:00,26,2023,2,Apples,5.64,5,Cash,Produce
2,170,2023-08-18,06:00:00,33,2023,3,Apples,17.92,1,Mobile Payment,Produce
3,86,2023-08-18,09:00:00,33,2023,4,Pasta,19.14,2,Mobile Payment,Grains
4,178,2023-08-06,02:00:00,32,2023,5,Rice,0.76,4,Debit Card,Grains
5,87,2023-07-30,09:00:00,31,2023,6,Chickpeas,11.30,1,Debit Card,Vegetarian


In [18]:
# Create a Sales column
combined <-
combined %>%
mutate(Sales = PriceUSD * Quantity) 

# Calculate the total sales for each week-year combination
weekly_sales <-
combined %>%
group_by(Week, Year) %>%
summarise(sales_per_week = sum(Sales)) %>%
ungroup()

# Calculate the mean weekly sales over the entire period
mean_weekly_sales = mean(weekly_sales$sales_per_week)

# Find the week with the smallest absolute sales deviation from the mean weekly sales
smallest_sales_deviation <- 
weekly_sales %>%
mutate(sales_dev =  abs(sales_per_week - mean_weekly_sales)) %>%
arrange(sales_dev) %>%
slice(1) %>%
pull(Week) %>%
as.integer()

smallest_sales_deviation

`summarise()` has grouped output by 'Week'. You can override using the
`.groups` argument.


[1] 24

In [19]:
# Step 2: What hour of the day (as a number on the 24-Hour scale) had the most hourly total sales?

# Convert Time from character to hms format and extract Hour, Minute, and Seconds
combined <-
combined %>%
mutate(Time = as_hms(Time)) %>%
mutate(Hour = hour(Time)) %>%
mutate(Minute = minute(Time)) %>%
mutate(Second = second(Time)) %>%
relocate(Hour, Minute, Second, .after = Time)

# Calculate the total sales for each hour and get the hour with the most sales
most_hourly_sales <-
combined %>%
group_by(Hour) %>%
summarise(total_sales = sum(Sales)) %>%
arrange(desc(total_sales)) %>%
slice(1) %>%
pull(Hour) %>%
as.integer()

most_hourly_sales

[1] 22

In [20]:
# Step 3: How many days passed between the three purchases of cornflakes by CustomerID 107, specifically the number of days between the first and second purchases, and the second and third purchases?

# Select the relevant columns, filter for CustomerID 107 and ProductName Cornflakes, and calculate the difference between each date and the previous date.
id7_cornflakes <- 
combined %>%
select(CustomerID, ProductName, Date) %>%
filter(CustomerID == 107, ProductName == 'Cornflakes') %>%
arrange(Date) %>%
mutate(diff = Date - lag(Date))

# Extract the second and third values from the diff column and store them in a vector called cornflakes_days
cornflakes_days <- c(id7_cornflakes$diff[2], id7_cornflakes$diff[3])
cornflakes_days

Time differences in days
[1]  6 40